# ETH Transaction Graph — Global & Daily Top Node Ranking

**Part of:** Ethereum Topological Anomaly Detection (ETH-TAD)  
**Paper:** Ofori-Boateng et al. (2021) - arXiv:2106.01806

## Description

Processes ETH and ERC20 transaction data (weekly parquet chunks), applies user-defined
filters, builds a weighted graph, and offers two ranking modes:

**Mode A — Edge-based ranking**  
Selects the top-N edges by weight and collects the nodes connected to those edges.

**Mode B — Centrality-based ranking**  
Ranks nodes directly by a graph centrality metric (default: PageRank).
Outputs the top-N central nodes per filter per period.

## Prerequisites
- Completed notebooks: 1a, 1b (data download)
- Required data: ETH and ERC20 transaction parquet files

## Outputs
- `ranking/global_top_nodes.parquet` — global rankings (both modes, all filters)
- `ranking/daily/daily_ranking_YYYY-MM_WN.parquet` — edge-based daily rankings
- `ranking/daily/daily_centrality_YYYY-MM_WN.parquet` — centrality daily rankings

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

import pandas as pd
from pathlib import Path

from ranking_functions import (
    check_directories,
    run_global_ranking,
    run_daily_ranking,
    run_global_centrality_ranking,
    run_daily_centrality_ranking,
)

print('Imports OK ✓')

## Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════

YEAR = 2020

ETH_DIR      = Path(f'data/{YEAR}/eth_tx_value_output/weekly')
ERC20_DIR    = Path(f'data/{YEAR}/erc20_tx_value_output/weekly')
RANKING_DIR  = Path(f'data/ranking/{YEAR}')
GLOBAL_FILE  = RANKING_DIR / 'global_top_nodes.parquet'
DAILY_DIR    = RANKING_DIR / 'daily'
INDEX_FILE   = RANKING_DIR / 'source_index.json'

RANKING_DIR.mkdir(parents=True, exist_ok=True)
DAILY_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════
# DATE RANGE
# ══════════════════════════════════════════════════════════════════════════

START_DATE = pd.Timestamp(f'{YEAR}-01-01')
END_DATE   = pd.Timestamp(f'{YEAR}-12-31')

# ══════════════════════════════════════════════════════════════════════════
# RANKING PARAMETERS
# ══════════════════════════════════════════════════════════════════════════

TOP_N = 1000  # How many top edges/nodes to consider per day/period

# ══════════════════════════════════════════════════════════════════════════
# DEAD ADDRESSES (to filter out)
# ══════════════════════════════════════════════════════════════════════════

DEAD_ADDRESSES = [
    None,
    "\\N",
]

# ══════════════════════════════════════════════════════════════════════════
# EDGE-BASED RANKING
# ══════════════════════════════════════════════════════════════════════════

METRICS   = ['tx_count', 'tx_value']
KEEP_COLS = ['date', 'from_addr', 'to_addr', 'tx_count', 'tx_value']

# ══════════════════════════════════════════════════════════════════════════
# CENTRALITY-BASED RANKING
# ══════════════════════════════════════════════════════════════════════════

CENTRALITY_WEIGHT_COL = 'tx_count'  # or 'tx_value'
SUBGRAPH_TOP_N = None  # e.g. 50_000 to prune before betweenness_approx

print(f'Year: {YEAR}')
print(f'Date range: {START_DATE.date()} → {END_DATE.date()}')
print(f'Top N: {TOP_N}')
print(f'Output: {RANKING_DIR}')
print('\nConfiguration set ✓')

## Filter Definitions

Define transaction filters. Each filter receives a DataFrame with an 'erc20' column  
(ETH-native rows have `erc20='ETH'`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BASIC FILTERS
# ══════════════════════════════════════════════════════════════════════════

FILTERS = {
    'contract_txs_ETH_only': lambda d: (d['tx_value'] == 0) & (d['erc20']=='ETH'),
    'simple_txs_ETH_only':   lambda d: (d['tx_value'] >  0) & (d['erc20']=='ETH'),
    'contract_txs_ALL': lambda d: (d['tx_value'] == 0),
    'simple_txs_ALL':   lambda d: (d['tx_value'] >  0),
    'contract_factory_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100),
    'contract_nonFactory_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']<100),
    'contract_highInput_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=500),
    'contract_mediumInput_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['total_input_bytes']<500)
}


# ══════════════════════════════════════════════════════════════════════════
# OPTIONAL: ERC20 TOKEN FILTERS
# ══════════════════════════════════════════════════════════════════════════

# Uncomment to add specific ERC20 token filters

# erc20_addresses = {
#     "USDT": "0xdac17f958d2ee523a2206206994597c13d831ec7",
#     "USDC": "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48",
#     "LINK": "0x514910771af9ca656af840dff83e8264ecf986ca",
#     "UNI":  "0x1f9840a85d5af5bf1d1762f925bdaddc4201f984",
#     "WBTC": "0x2260fac5e5542a773aa44fbcfedf7c193bc2c599",
#     "DAI":  "0x6b175474e89094c44da98b954eedeac495271d0f",
# }

# addresses_lower = {k: v.lower() for k, v in erc20_addresses.items()}

# erc20filters = {
#     symbol: (lambda addr: (lambda d: d["erc20"].str.lower() == addr))(address)
#     for symbol, address in addresses_lower.items()
# }

# # Merge with basic filters
# FILTERS = {**FILTERS, **erc20filters}

print(f'Filters defined: {list(FILTERS.keys())}')

## Sanity Check Data Directories

In [ ]:
assert check_directories(ETH_DIR, ERC20_DIR), 'Fix the folder paths above before continuing.'

## Run Rankings

Uncomment whichever runs you want. Order shown is recommended (cheapest first).

### Mode A: Edge-Based Ranking

In [ ]:
# Global edge-based ranking
# run_global_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     global_file=GLOBAL_FILE,
#     metrics=METRICS,
#     keep_cols=KEEP_COLS,
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
# )

In [ ]:
# Daily edge-based ranking
run_daily_ranking(
    filters=FILTERS,
    eth_dir=ETH_DIR,
    erc20_dir=ERC20_DIR,
    daily_dir=DAILY_DIR,
    index_file=INDEX_FILE,
    metrics=METRICS,
    keep_cols=KEEP_COLS,
    start=START_DATE,
    end=END_DATE,
    top_n=TOP_N,
    dead_ads=DEAD_ADDRESSES,
)

### Mode B: Centrality-Based Ranking

**Available centrality metrics:**
- `page_rank` — weighted PageRank (fast, default choice)
- `degree` — unweighted degree
- `strength` — weighted degree
- `k_core` — k-core number
- `hits_hub` — hub scores
- `hits_authority` — authority scores
- `eigenvector` — eigenvector centrality
- `clustering` — weighted clustering coefficient
- `betweenness_approx` — betweenness centrality (slow, use `subgraph_top_n`)

**Edge mode:** Append `_edge` to any metric (e.g., `page_rank_edge`) to rank edges instead of nodes.

In [ ]:
# Global centrality ranking — fast metrics (no subgraph needed)
# run_global_centrality_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     global_file=GLOBAL_FILE,
#     centrality_metrics=['page_rank', 'k_core', 'strength'],
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
#     weight_col=CENTRALITY_WEIGHT_COL,
#     subgraph_top_n=SUBGRAPH_TOP_N,
# )

In [ ]:
# Global centrality ranking — betweenness (MUST use subgraph_top_n)
# run_global_centrality_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     global_file=GLOBAL_FILE,
#     centrality_metrics=['betweenness_approx'],
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
#     weight_col=CENTRALITY_WEIGHT_COL,
#     subgraph_top_n=50_000,  # Required for betweenness
# )

In [ ]:
# Daily centrality ranking — fast metrics
# run_daily_centrality_ranking(
#     filters=FILTERS,
#     eth_dir=ETH_DIR,
#     erc20_dir=ERC20_DIR,
#     daily_dir=DAILY_DIR,
#     index_file=INDEX_FILE,
#     centrality_metrics=['page_rank', 'k_core', 'strength'],
#     start=START_DATE,
#     end=END_DATE,
#     top_n=TOP_N,
#     dead_ads=DEAD_ADDRESSES,
#     weight_col=CENTRALITY_WEIGHT_COL,
#     subgraph_top_n=SUBGRAPH_TOP_N,
# )

In [ ]:
# Daily centrality ranking — clustering coefficient
run_daily_centrality_ranking(
    filters=FILTERS,
    eth_dir=ETH_DIR,
    erc20_dir=ERC20_DIR,
    daily_dir=DAILY_DIR,
    index_file=INDEX_FILE,
    centrality_metrics=['clustering'],
    start=START_DATE,
    end=END_DATE,
    top_n=TOP_N,
    dead_ads=DEAD_ADDRESSES,
    weight_col=CENTRALITY_WEIGHT_COL,
    subgraph_top_n=10_000,
)

## Inspection

View the generated rankings.

In [ ]:
# Global ranking
if GLOBAL_FILE.exists():
    g = pd.read_parquet(GLOBAL_FILE)
    print(f'Global ranking shape: {g.shape}')
    display(g.head(10))
    display(
        g.groupby(['filter', 'ranking_metric', 'start_date', 'end_date'])
         .size().rename('node_count').reset_index()
    )
else:
    print('No global ranking file found.')

In [ ]:
# Daily ranking
daily_files = sorted(DAILY_DIR.glob('*.parquet'))
print(f'Daily ranking files: {len(daily_files)}')

if daily_files:
    sample = pd.read_parquet(daily_files[0])
    print(f'\nSample file: {daily_files[0].name}  ({len(sample):,} rows)')
    display(sample.head(10))

    frames   = [pd.read_parquet(f)[['filter', 'ranking_metric', 'date']] for f in daily_files]
    coverage = pd.concat(frames, ignore_index=True)
    coverage['date'] = pd.to_datetime(coverage['date'])
    print('\nDate coverage per (filter, metric):')
    display(
        coverage.groupby(['filter', 'ranking_metric'])
                .agg(dates=('date', 'nunique'), min_date=('date', 'min'), max_date=('date', 'max'))
                .reset_index()
    )